In [34]:
!pip install -q openenv-core

import sys
from pathlib import Path
# Add repo root so we can import compressionenv (parent of compressionenv/ or cwd)
_repo = Path.cwd()
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))

from compressionenv import CompressionenvAction, CompressionenvObservation, CompressionenvEnv, ensure_essays
from compressionenv.server.compressionenv_environment import CompressionenvEnvironment

print("Package loaded.")

All definitions loaded.


In [35]:
# --- Upload essays to remote environment ---
# Upload essays.tar.gz alongside this notebook, then run this cell.
ensure_essays()

essays/ already present (229 files)


In [36]:
# --- Start server ---
import threading
import time
import uvicorn

from compressionenv.server.app import app, _show_port_processes, _kill_port

_show_port_processes(8000)
_kill_port(8000)
time.sleep(1)

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
time.sleep(1)
print("Server started on http://0.0.0.0:8000")
_show_port_processes(8000)

No processes found on port 8000 (or lsof/ss not available)


INFO:     Started server process [117]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Server started on http://0.0.0.0:8000
No processes found on port 8000 (or lsof/ss not available)


In [37]:
# --- Run single worker (identity codec) ---
from compressionenv.worker import _run_one_trajectory

BASE_URL = "http://localhost:8000"
traj = _run_one_trajectory(0, BASE_URL)
print(f"essay_id: {traj['essay_id']}, steps: {len(traj['steps'])}")
for s in traj["steps"]:
    o = s["observation"]
    print(f"  step={s['step']:>2}  valid={o.get('valid')}  compressed={o.get('compressed_size_bytes')}  reward={s['reward']}  done={s['done']}")

essay_id: webstartups, steps: 21
  step= 0  valid=True  compressed=None  reward=0.0  done=False
  step= 1  valid=True  compressed=19659  reward=0.0  done=False
  step= 2  valid=True  compressed=19659  reward=0.0  done=False
  step= 3  valid=True  compressed=19659  reward=0.0  done=False
  step= 4  valid=True  compressed=19659  reward=0.0  done=False
  step= 5  valid=True  compressed=19659  reward=0.0  done=False
  step= 6  valid=True  compressed=19659  reward=0.0  done=False
  step= 7  valid=True  compressed=19659  reward=0.0  done=False
  step= 8  valid=True  compressed=19659  reward=0.0  done=False
  step= 9  valid=True  compressed=19659  reward=0.0  done=False
  step=10  valid=True  compressed=19659  reward=0.0  done=False
  step=11  valid=True  compressed=19659  reward=0.0  done=False
  step=12  valid=True  compressed=19659  reward=0.0  done=False
  step=13  valid=True  compressed=19659  reward=0.0  done=False
  step=14  valid=True  compressed=19659  reward=0.0  done=False
  step=1

In [38]:
# --- Run 4 workers in parallel and save ---
import json
import multiprocessing
import os

from compressionenv.worker import _run_one_trajectory, run_worker_process

NUM_WORKERS, OUTPUT_PATH = 4, "output/trajectories.json"
with multiprocessing.Pool(NUM_WORKERS) as pool:
    results = pool.map(run_worker_process, [(i, BASE_URL) for i in range(NUM_WORKERS)])

out_data = {f"worker_{wid}": t for wid, t in results}
for wk, t in out_data.items():
    print(f"{wk}: essay_id={t['essay_id']}, steps={len(t['steps'])}")

os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)
with open(OUTPUT_PATH, "w") as f:
    json.dump(out_data, f, indent=2)
print(f"\nWrote trajectories to {OUTPUT_PATH}")

/opt/conda/lib/python3.13/multiprocessing/popen_fork.py:67: DeprecationWarning: This process (pid=117) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


worker_0: essay_id=vb, steps=21
worker_1: essay_id=inequality, steps=21
worker_2: essay_id=smart, steps=21
worker_3: essay_id=webstartups, steps=21

Wrote trajectories to output/trajectories.json


In [39]:
# --- LLM agent (Qwen 8B) ---
# Docs: https://huggingface.co/Qwen/Qwen3-8B
# Model loads lazily on first generate_compression_action call.
!pip install -q transformers torch accelerate

from compressionenv.llm_agent import generate_compression_action, llm_generate

# Optional: quick test to trigger model load
# llm_generate("Say 'ready' in one word.", max_new_tokens=10)
print("LLM agent ready.")

/opt/conda/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=117) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [41]:
# --- Run single worker with LLM-generated actions ---
# Requires: cells 0,1,2,5 run first (definitions, essays, server, LLM agent)
import os

from compressionenv.worker import _run_trajectory_llm

BASE_URL = "http://localhost:8000"
TRAJECTORY_FILE = "/home/jovyan/llm_trajectory_steps.txt"

traj = _run_trajectory_llm(BASE_URL, trajectory_file=TRAJECTORY_FILE)
print(f"essay_id: {traj['essay_id']}, steps: {len(traj['steps'])}")
print(f"\nTo download the trajectory file locally, run:")
print(f"npx @northflank/cli download service file --projectId hackathon --serviceId jupyter-pytorch --localPath llm_trajectory_steps.txt --remotePath {TRAJECTORY_FILE}")

essay_id: future  baselines: {'zlib': 8967, 'bz2': 7938, 'lzma': 8640}  best: 7938 bytes
  step= 0  valid=True  compressed=None  reward=0.0  done=False
  step= 1  valid=False  compressed=None  reward=-1.0  done=False
  step= 2  valid=False  compressed=None  reward=-1.0  done=False
  step= 3  valid=False  compressed=None  reward=-1.0  done=False
  step= 4  valid=False  compressed=None  reward=-1.0  done=False
  step= 5  valid=False  compressed=None  reward=-1.0  done=False
  step= 6  valid=False  compressed=None  reward=-1.0  done=False
  step= 7  valid=False  compressed=None  reward=-1.0  done=False
  step= 8  valid=False  compressed=None  reward=-1.0  done=False
  step= 9  valid=False  compressed=None  reward=-1.0  done=False
  step=10  valid=False  compressed=None  reward=-1.0  done=False
  step=11  valid=False  compressed=None  reward=-1.0  done=False
  step=12  valid=False  compressed=None  reward=-1.0  done=False
  step=13  valid=False  compressed=None  reward=-1.0  done=False
  s